In [ ]:
import glob              
import pandas as pd

DATA = "../data"

# --- CRSP names file ---
names = pd.read_csv(glob.glob(f"{DATA}/honhxq*")[0])
names.columns = names.columns.str.lower()
start_col = "secinfostartdt" if "secinfostartdt" in names else "namedt"
end_col = "secinfoenddt" if "secinfoenddt" in names else "nameenddt"
names["ticker"] = names["ticker"].astype(str).str.upper().str.strip()
names[start_col] = pd.to_datetime(names[start_col], errors="coerce")
names[end_col] = pd.to_datetime(names[end_col], errors="coerce").fillna(pd.Timestamp("2100-01-01"))
names = names[["permno", "ticker", start_col, end_col]].drop_duplicates()

def match_permno(df, date_col, tol_days=0, ticker_col="ticker"):
    """Match ticker to PERMNO whose ticker window contains the date (+/- tolerance)."""
    d = df.reset_index(names="row_id").merge(
        names, left_on=ticker_col, right_on="ticker", how="left", suffixes=("", "_crsp"))
    lo = d[start_col] - pd.Timedelta(days=tol_days)
    hi = d[end_col] + pd.Timedelta(days=tol_days)
    d = d[(d[date_col] >= lo) & (d[date_col] <= hi)]
    d = d.sort_values(end_col).drop_duplicates("row_id", keep="last")  # latest window if ambiguous
    out = df.copy()
    out["permno"] = d.set_index("row_id")["permno"].reindex(out.index)
    return out

add = pd.read_csv(f"{DATA}/sp500_additions_2023_2025.csv", parse_dates=["effective_date"])
rem = pd.read_csv(f"{DATA}/sp500_removals_2023_2025.csv", parse_dates=["effective_date"])

# removed companies are often delisted just before the effective date -> small tolerance
add = match_permno(add, "effective_date", tol_days=10)
rem = match_permno(rem, "effective_date", tol_days=60)

print("Additions without PERMNO:\n", add.loc[add.permno.isna(), ["effective_date", "ticker", "company"]])
print("Removals without PERMNO:\n", rem.loc[rem.permno.isna(), ["effective_date", "ticker", "company"]])

Additions without PERMNO:
 Empty DataFrame
Columns: [effective_date, ticker, company]
Index: []
Removals without PERMNO:
 Empty DataFrame
Columns: [effective_date, ticker, company]
Index: []


In [7]:
old = pd.read_csv(f"{DATA}/sp500_constituent_changes.csv")
old["SP500 Start"] = pd.to_datetime(old["SP500 Start"])
old["SP500 End"] = pd.to_datetime(old["SP500 End"])
old = old.rename(columns={"PERMNO": "permno", "Ticker": "ticker", "Company": "company",
                          "SP500 Start": "start", "SP500 End": "end"})

LAST_OLD = old["end"].max()              # 2022-12-30: spells ending here are still open
END_NEW = pd.Timestamp("2025-12-31")     # end of our CRSP sample

# 1) close old open spells for companies removed 2023-2025, otherwise extend to END_NEW
rem_dates = rem.dropna(subset=["permno"]).groupby("permno")["effective_date"].min()
is_open = old["end"] >= LAST_OLD
old.loc[is_open, "end"] = (old.loc[is_open, "permno"].map(rem_dates)
                           .fillna(END_NEW).values)
old["source"] = "wrds_file"

# 2) new spells from additions (closed if removed later within the sample)
new = add.dropna(subset=["permno"]).copy()
new = new.rename(columns={"effective_date": "start"})
new["end"] = new["permno"].map(rem_dates).where(lambda s: s > new["start"]).fillna(END_NEW)
new["source"] = "manual_2023_2025"

cols = ["permno", "ticker", "company", "start", "end", "source"]
spells = pd.concat([old[cols], new[cols + [] ]], ignore_index=True)
spells["permno"] = spells["permno"].astype("Int64")
spells = spells.drop_duplicates(["permno", "start", "end"]).sort_values(["permno", "start"])

# 3) event table = additions with metadata (keep spin-offs flagged, filter later)
events = add.dropna(subset=["permno"]).rename(columns={"effective_date": "effective_date"})
old_add = old[old["start"] >= "1976-01-01"][["permno", "ticker", "company", "start"]].copy()
old_add = old_add.rename(columns={"start": "effective_date"})
old_add["event_type"] = "unknown"   # WRDS file has no event type: classify later
events = pd.concat([old_add, events[["permno", "ticker", "company", "effective_date", "event_type",
                                     "migration_from_midcap400", "announcement_date"]]],
                   ignore_index=True)

for name, df in [("sp500_spells", spells), ("sp500_events", events)]:
    try:
        df.to_parquet(f"{DATA}/{name}.parquet", index=False)
    except ImportError:
        df.to_csv(f"{DATA}/{name}.csv", index=False)

print(spells.groupby("source").size())
print(spells["end"].max(), events["effective_date"].dt.year.value_counts().sort_index().tail(5))

source
manual_2023_2025      49
wrds_file           1889
dtype: int64
2025-12-31 00:00:00 effective_date
2021    19
2022    17
2023    15
2024    16
2025    18
Name: count, dtype: int64
